In [ ]:
# ===== NIGHT RUN — CELL 0 : setup, cache আবিষ্কার, embedding load =====
# সম্পূর্ণ স্বয়ংসম্পূর্ণ notebook। Save & Run All দিয়ে চালিয়ে ঘুমাতে যাও।
# Input: (১) astro-essentials  (২) astro-img384  (৩) siglip dataset [ঐচ্ছিক]
import os, glob, re, time, json, gc
import numpy as np, pandas as pd

OUT    = '/kaggle/working'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']
T00    = time.time()

W    = os.path.dirname(glob.glob('/kaggle/input/**/emb_clip_img.npy', recursive=True)[0])
IMGD = os.path.dirname(glob.glob('/kaggle/input/**/*.jpg', recursive=True)[0])
print('W    =', W)
print('IMGD =', IMGD, '| jpgs:', len(os.listdir(IMGD)))

mtr = pd.read_parquet(f'{W}/meta_train.parquet')
mte = pd.read_parquet(f'{W}/meta_test.parquet')
for d in (mtr, mte):
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr.label.astype(str)
texts = pd.read_parquet(f'{W}/texts.parquet')

def center(x):
    z = x - x.mean(0, keepdims=True)
    return z / np.linalg.norm(z, axis=1, keepdims=True).clip(1e-8)

img_ix = {h: i for i, h in enumerate(pd.read_parquet(f'{W}/img_hashes.parquet').hash.values)}
txt_ix = {h: i for i, h in enumerate(pd.read_parquet(f'{W}/txt_hashes.parquet').hash.values)}

# space → (image matrix, text matrix, img index, txt index)
SP = {
 'clip': (np.load(f'{W}/emb_clip_img.npy'),          np.load(f'{W}/emb_clip_txt.npy'),          img_ix, txt_ix),
 'sci' : (None,                                      center(np.load(f'{W}/emb_sci_txt.npy')),   None,   txt_ix),
 'dino': (center(np.load(f'{W}/emb_dino_img.npy')),  None,                                      img_ix, None),
}

# SigLIP attach করা থাকলে নিজে থেকেই ঢুকে পড়বে — image+text-এ এটাই CLIP-কে হারিয়েছে
sg = glob.glob('/kaggle/input/**/emb_siglip_img.npy', recursive=True)
if sg:
    S = os.path.dirname(sg[0])
    si = {h: i for i, h in enumerate(pd.read_parquet(f'{S}/img_hashes_siglip.parquet').hash.values)}
    st = {h: i for i, h in enumerate(pd.read_parquet(f'{S}/txt_hashes_siglip.parquet').hash.values)}
    SP['sig'] = (np.load(f'{S}/emb_siglip_img.npy'), np.load(f'{S}/emb_siglip_txt.npy'), si, st)
    print('SigLIP পাওয়া গেছে →', S)
else:
    print('SigLIP নেই — CLIP/SciNCL/DINO দিয়েই চলবে')

SPACES = {'image+text':  ['clip'] + (['sig'] if 'sig' in SP else []),
          'text+text':   ['clip','sci'] + (['sig'] if 'sig' in SP else []),
          'image+image': ['clip','dino'] + (['sig'] if 'sig' in SP else [])}
print('spaces:', SPACES)

def side(hashes, types, space):
    """এক পাশের embedding matrix — dim encoder থেকেই নেয়, hardcode নয়"""
    ie, te_, ii, ti = SP[space]
    dim = (ie if ie is not None else te_).shape[1]
    out = np.zeros((len(hashes), dim), dtype=np.float32)
    for i, (h, t) in enumerate(zip(hashes, types)):
        if t == 'image' and ie is not None: out[i] = ie[ii[h]]
        elif t == 'text' and te_ is not None: out[i] = te_[ti[h]]
    return out

def pair_block(e1, e2, tag):
    n = e1.shape[1]
    X = np.hstack([np.abs(e1-e2), e1*e2])
    return X, [f'{tag}_d{i}' for i in range(n)] + [f'{tag}_p{i}' for i in range(n)]

print(f'setup ok  {time.time()-T00:.0f}s')

# hang হলে নিজে থেকেই stack trace ছাপবে — আর কখনো অন্ধ অপেক্ষা করতে হবে না
import faulthandler, sys
faulthandler.dump_traceback_later(600, repeat=True, file=sys.stderr)


In [ ]:
# ===== OCR — thread-based, fork নেই, সময়সীমা আছে =====
# আগের সংস্করণ multiprocessing.Pool ব্যবহার করত। cell 0-এ numpy/OpenMP-র thread
# pool চালু হয়ে যায়; সেই অবস্থায় fork() করলে child libgomp-এর lock-এ চিরতরে
# আটকে যায় আর parent অপেক্ষা করতেই থাকে — শূন্য output, অনন্ত hang।
# ThreadPool fork করে না। tesseract নিজেই আলাদা subprocess, তাই GIL সমস্যা নেই।
import os, sys
os.environ['OMP_THREAD_LIMIT'] = '1'      # tesseract নিজেই core-সংখ্যক thread খোলে
os.environ['OMP_NUM_THREADS']  = '1'

OCRP   = f'{OUT}/ocr.parquet'
BUDGET = 75 * 60        # এর বেশি নয় — আটকে গেলেও বাকি notebook চলবে
EVERY  = 200

try:
    import pytesseract
    from PIL import Image
    print('tesseract', pytesseract.get_tesseract_version(), flush=True)
    HAVE_OCR = True
except Exception as e:
    print('⚠️ tesseract নেই:', repr(e)[:120], flush=True); HAVE_OCR = False

hashes = sorted(h[:-4] for h in os.listdir(IMGD))
done = {}
for src in glob.glob('/kaggle/input/**/ocr.parquet', recursive=True) + [OCRP]:
    if os.path.exists(src):
        prev = pd.read_parquet(src); done.update(dict(zip(prev.hash, prev.ocr)))
        print('আগের OCR পাওয়া গেছে:', src, len(done), flush=True)
        break

CFG  = '--oem 1 --psm 6'   # psm 6 = uniform block; psm 11-এর চেয়ে অনেক দ্রুত
KEEP = re.compile(r'[A-Za-z0-9][A-Za-z0-9\.\-\+/]{1,}')

def ocr_one(h):
    try:
        im = Image.open(f'{IMGD}/{h}.jpg').convert('L')
        raw = pytesseract.image_to_string(im, config=CFG, timeout=15)
        toks = [t for t in KEEP.findall(raw) if (not t.isdigit()) or len(t) >= 4]
        return h, ' '.join(toks[:120])
    except Exception:
        return h, ''

def flushp():
    pd.DataFrame({'hash': list(done), 'ocr': list(done.values())}).to_parquet(OCRP)

if HAVE_OCR:
    todo = [h for h in hashes if h not in done]
    print('বাকি:', len(todo), flush=True)
    t0 = time.time()
    if todo:
        # আগে একটা image একা — এখানেই আটকালে সমান্তরাল করে কোনো লাভ নেই
        h0, s0 = ocr_one(todo[0]); done[h0] = s0; todo = todo[1:]
        dt1 = time.time() - t0
        print(f'একটা image: {dt1:.2f}s | আনুমানিক মোট {dt1*len(todo)/4/60:.0f} মিনিট', flush=True)
        print(f'নমুনা: {s0[:120]}', flush=True)

        from multiprocessing.pool import ThreadPool     # thread, process নয় → fork নেই
        pool = ThreadPool(4)
        try:
            for i, (h, s) in enumerate(pool.imap_unordered(ocr_one, todo, chunksize=4), 1):
                done[h] = s
                if i % EVERY == 0:
                    el = time.time() - t0
                    print(f'  {i}/{len(todo)}  {el:.0f}s  ({el/i:.2f}s/img)  '
                          f'বাকি ~{(len(todo)-i)*el/i/60:.0f}m', flush=True)
                    if i % 1000 == 0: flushp()
                if time.time() - t0 > BUDGET:
                    print(f'⏱️ {BUDGET//60} মিনিট শেষ — যা হয়েছে তাই নিয়ে এগোচ্ছি', flush=True)
                    break
        finally:
            pool.terminate(); pool.join()

flushp()
ln = pd.Series([len(v.split()) for v in done.values()]) if done else pd.Series([0])
print(f'\nOCR: {len(done)}/{len(hashes)} | token/image mean {ln.mean():.1f} '
      f'median {ln.median():.0f} | খালি {int((ln==0).sum())}', flush=True)
# GATE: mean < 5 হলে OCR feature দুর্বল, কিন্তু পরের cell তবু চলবে


In [ ]:
# ===== NIGHT RUN — CELL 2 : feature builder =====
# তিনটা নতুন block, তিনটাই আলাদা রোগের ওষুধ:
#   (ক) IDF token overlap — same_figure-এর প্রায়-নির্ধারক সূত্র
#   (খ) TF-IDF cosine     — OCR-এর পর image-ও text, তাই তিন subset-এই চলে
#   (গ) hubness z + rank   — cosine absolute, অথচ সীমানাটা relative (hub object সবার কাছেই কাছে)
from sklearn.feature_extraction.text import TfidfVectorizer

texts_map = dict(zip(texts.hash, texts.text))
_o = pd.read_parquet(OCRP) if os.path.exists(OCRP) else pd.DataFrame({'hash':[], 'ocr':[]})
ocr_map = dict(zip(_o.hash, _o.ocr))

ap = pd.concat([mtr[['h1','t1','h2','t2']], mte[['h1','t1','h2','t2']]])
typ = dict(zip(pd.concat([ap.h1, ap.h2]), pd.concat([ap.t1, ap.t2])))
docs = {h: (texts_map.get(h,'') if t == 'text' else ocr_map.get(h,'')) for h, t in typ.items()}
corpus_h = list(docs)

tfv  = TfidfVectorizer(lowercase=True, token_pattern=r'[A-Za-z0-9][A-Za-z0-9\.\-\+/]+',
                       min_df=2, max_features=60000, sublinear_tf=True)
Tm   = tfv.fit_transform([docs[h] for h in corpus_h])
tpos = {h: i for i, h in enumerate(corpus_h)}
idf  = dict(zip(tfv.get_feature_names_out(), tfv.idf_))
TOK  = re.compile(r'[A-Za-z0-9][A-Za-z0-9\.\-\+/]+')
tokset = {h: set(t.lower() for t in TOK.findall(docs[h])) for h in corpus_h}
print('tfidf:', Tm.shape, '| গড় token/object:', np.mean([len(v) for v in tokset.values()]).round(1))

def text_block(d):
    A = Tm[[tpos[h] for h in d.h1]]; B = Tm[[tpos[h] for h in d.h2]]
    cos = np.asarray(A.multiply(B).sum(1)).ravel()
    n_sh, idf_sh, jac, rare, l1, l2 = [], [], [], [], [], []
    for h1, h2 in zip(d.h1.values, d.h2.values):
        a, b = tokset[h1], tokset[h2]
        inter = a & b
        w = [idf.get(t, 0.0) for t in inter]
        n_sh.append(len(inter)); idf_sh.append(float(np.sum(w)))
        jac.append(len(inter)/max(len(a|b), 1))
        rare.append(float(np.sum([x for x in w if x > 6.0])))   # শুধু বিরল token — identifier
        l1.append(len(a)); l2.append(len(b))
    X = np.c_[cos, n_sh, idf_sh, jac, rare, np.minimum(l1,l2), np.maximum(l1,l2)]
    return X.astype(np.float32), ['tf_cos','tok_n','tok_idf','tok_jac','tok_rare','tok_lmin','tok_lmax']

rng = np.random.default_rng(0)
def hub_block(e1, e2, tag, nref=1500):
    cos = (e1*e2).sum(1)
    cs = []
    for e_q, e_pool in ((e1, e2), (e2, e1)):
        R  = e_pool[rng.choice(len(e_pool), min(nref, len(e_pool)), replace=False)]
        S  = e_q @ R.T
        cs += [(cos - S.mean(1)) / S.std(1).clip(1e-6), (S < cos[:, None]).mean(1)]
    z1, r1, z2, r2 = cs
    X = np.c_[cos, np.linalg.norm(e1-e2, axis=1),
              np.minimum(z1,z2), np.maximum(z1,z2),
              np.minimum(r1,r2), np.maximum(r1,r2)]      # min/max → swap-invariant
    return X.astype(np.float32), [f'{tag}_cos', f'{tag}_l2', f'{tag}_zmin', f'{tag}_zmax',
                                  f'{tag}_rmin', f'{tag}_rmax']

def build2(m, combo):
    d = m[m.combo == combo].reset_index(drop=True).copy()
    if combo == 'image+text':          # canonical: t1 সবসময় image
        sw = d.t1.values == 'text'
        for a, b in (('h1','h2'), ('t1','t2'), ('len1','len2')):
            d.loc[sw, [a, b]] = d.loc[sw, [b, a]].values
    blocks, cols = [], []
    for sp in SPACES[combo]:
        e1, e2 = side(d.h1.values, d.t1.values, sp), side(d.h2.values, d.t2.values, sp)
        for fn in (pair_block, hub_block):
            X, c = fn(e1, e2, sp); blocks.append(X); cols += c
    X, c = text_block(d); blocks.append(X); cols += c
    mn = np.minimum(d.len1, d.len2).values; mx = np.maximum(d.len1, d.len2).values
    blocks.append(np.c_[mn, mx, mx/np.maximum(mn,1)].astype(np.float32))
    cols += ['len_min','len_max','len_ratio']
    return d, np.hstack(blocks).astype(np.float32), cols

FEAT = {}
for c in SPACES:
    FEAT[c] = (build2(mtr, c), build2(mte, c))     # একবারই বানাই, পরে বারবার নয়
    print(f'{c:12s} train {FEAT[c][0][1].shape}  test {FEAT[c][1][1].shape}', flush=True)
print(f'features ok  {time.time()-T00:.0f}s')


In [ ]:
# ===== NIGHT RUN — CELL 3 : LGBM + MLP + threshold → OOF, submission =====
# এক পাসে: subset-প্রতি দুই model, blend weight ও per-class multiplier OOF-এ tune,
# তারপর একই সেটিংয়ে test predict।
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline

def coord_ascent(P, ytrue, classes, rounds=10):
    """argmax macro-F1-এর জন্য optimal নয় — per-class multiplier greedy খুঁজি"""
    w = np.ones(len(classes))
    best = f1_score(ytrue, [classes[i] for i in (P*w).argmax(1)], average='macro')
    for _ in range(rounds):
        moved = False
        for j in range(len(classes)):
            for m in (0.7,0.85,0.95,1.05,1.15,1.35):
                w2 = w.copy(); w2[j] *= m
                s = f1_score(ytrue, [classes[i] for i in (P*w2).argmax(1)], average='macro')
                if s > best + 1e-5: best, w, moved = s, w2, True
        if not moved: break
    return w, best

sub = pd.DataFrame({'id': mte.id.values})
for c in LABELS: sub[c] = 0
report, sizes = {}, {}

for combo in ['image+text','text+text','image+image']:
    (d, X, cols), (dt, Xt, _) = FEAT[combo]
    classes = sorted(d.y.unique()); K = len(classes)
    y = d.y.map({c:i for i,c in enumerate(classes)}).values
    oof_l = np.zeros((len(y),K)); oof_m = np.zeros((len(y),K))
    te_l  = np.zeros((len(dt),K)); te_m = np.zeros((len(dt),K))

    for f, (tr_i, va_i) in enumerate(StratifiedKFold(5, shuffle=True, random_state=0).split(X, y)):
        Xtr, ytr = X[tr_i], y[tr_i]   # feature ইতিমধ্যে swap-invariant → duplication বৃথা
        g = lgb.LGBMClassifier(objective='multiclass', num_class=K, n_estimators=500,
                               learning_rate=0.05, num_leaves=31, colsample_bytree=0.3,
                               subsample=0.8, subsample_freq=1, class_weight='balanced',
                               verbose=-1, n_jobs=-1).fit(Xtr, ytr)
        oof_l[va_i] = g.predict_proba(X[va_i]); te_l += g.predict_proba(Xt)/5

        # MLP: tree একটা করে dimension ভাগ করে; embedding-এর তথ্য linear combination-এ
        m = make_pipeline(StandardScaler(), PCA(n_components=192, random_state=0),
                          MLPClassifier((256,128), alpha=1e-3, max_iter=400,
                                        early_stopping=True, n_iter_no_change=20,
                                        random_state=0)).fit(X[tr_i], y[tr_i])
        oof_m[va_i] = m.predict_proba(X[va_i]); te_m += m.predict_proba(Xt)/5
        print(f'  {combo} fold {f+1}/5  {time.time()-T00:.0f}s', flush=True)

    f1 = lambda P: f1_score(d.y, [classes[i] for i in P.argmax(1)], average='macro')
    bw, bs = max(((w, f1((1-w)*oof_l + w*oof_m)) for w in np.arange(0,1.01,0.1)), key=lambda t: t[1])
    bo, bt = (1-bw)*oof_l + bw*oof_m, (1-bw)*te_l + bw*te_m
    mult, s_thr = coord_ascent(bo, d.y.values, classes)
    pred = [classes[i] for i in (bo*mult).argmax(1)]

    print(f'\n### {combo}:  lgbm {f1(oof_l):.4f} | mlp {f1(oof_m):.4f} | '
          f'blend(w={bw:.1f}) {bs:.4f} | +threshold {s_thr:.4f}')
    print(classification_report(d.y, pred, digits=3))
    print(pd.DataFrame(confusion_matrix(d.y, pred, labels=classes), index=classes, columns=classes))
    report[combo] = {'lgbm': round(f1(oof_l),4), 'mlp': round(f1(oof_m),4), 'w_mlp': round(float(bw),2),
                     'blend': round(float(bs),4), 'blend_thr': round(float(s_thr),4),
                     'mult': [round(x,3) for x in mult], 'classes': classes}
    sizes[combo] = len(d)

    pos = pd.Index(sub.id).get_indexer(dt.id.values)
    assert (sub.loc[pos,'id'].values == dt.id.values).all()
    for i, j in enumerate((bt*mult).argmax(1)):
        sub.loc[pos[i], classes[j]] = 1
    np.save(f'{OUT}/oof_{combo.replace("+","_")}.npy', bo)
    np.save(f'{OUT}/te_{combo.replace("+","_")}.npy',  bt)

sub[['id']+LABELS].to_csv(f'{OUT}/submission.csv', index=False)
tot = sum(sizes.values())
report['overall_blend']     = round(sum(report[c]['blend']*sizes[c]     for c in sizes)/tot, 4)
report['overall_blend_thr'] = round(sum(report[c]['blend_thr']*sizes[c] for c in sizes)/tot, 4)
report['baseline']          = 0.4806
json.dump(report, open(f'{OUT}/report.json','w'), indent=1)

print('\n' + '='*60)
print(json.dumps(report, indent=1))
print('\nsubmission:'); print(sub[LABELS].sum().to_dict())
print('প্রতি row-তে ঠিক একটা 1:', sub[LABELS].sum(1).value_counts().to_dict())
print(f'\nমোট সময় {(time.time()-T00)/60:.1f} মিনিট')
